# Bronze Layer — Raw Data Ingestion

**Notebook:** `01_bronze_raw_ingestion`  
**Layer:** Bronze (Raw)  
**Lakehouse:** `RetailSalesLakehouse`  
**DP-700 Skill:** Ingest and Transform Data > Batch Data Processing

## Purpose
This notebook ingests raw source data from CSV files into the Bronze layer of the Lakehouse.
No business transformations are applied — data is stored as-is in Delta format, preserving full fidelity.

## Design Principles
- Schema-on-read: data written as-is from source
- Append-only with ingestion metadata columns
- Full + incremental load patterns supported via `load_mode` parameter
- Source lineage tracked via `_source_file` and `_ingestion_timestamp` columns

## Parameters
These parameters are injected by the Data Pipeline at runtime.

In [ ]:
# Pipeline Parameters (injected at runtime via Data Pipeline)
load_mode = "full"  # Options: 'full' | 'incremental'
source_folder = "Files/raw/"  # OneLake path to source CSV files
watermark_date = "1900-01-01"  # Used for incremental loads
run_id = "manual"  # Pipeline run ID for auditing

## Setup: Imports and Configuration

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, current_timestamp, input_file_name,
    to_timestamp, when, trim, upper
)
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    DoubleType, BooleanType, DateType, TimestampType
)
from datetime import datetime

# Spark session is pre-configured in Fabric notebooks
spark = SparkSession.builder.getOrCreate()

# Configuration
LAKEHOUSE_NAME = "RetailSalesLakehouse"
BRONZE_SCHEMA = "bronze"
INGESTION_TS = datetime.utcnow().isoformat()

print(f"Load Mode: {load_mode}")
print(f"Run ID: {run_id}")
print(f"Ingestion Timestamp: {INGESTION_TS}")

## Helper Functions

In [ ]:
def add_audit_columns(df, source_table: str):
    """Add standard audit columns to track lineage and ingestion metadata."""
    return (
        df
        .withColumn("_source_table", lit(source_table))
        .withColumn("_source_file", input_file_name())
        .withColumn("_ingestion_timestamp", current_timestamp())
        .withColumn("_pipeline_run_id", lit(run_id))
        .withColumn("_load_mode", lit(load_mode))
    )

def write_to_bronze(df, table_name: str, mode: str = "overwrite"):
    """Write DataFrame to Bronze Delta table in the Lakehouse."""
    full_table = f"{BRONZE_SCHEMA}.{table_name}"
    (
        df.write
        .format("delta")
        .mode(mode)
        .option("overwriteSchema", "true")
        .saveAsTable(full_table)
    )
    row_count = spark.table(full_table).count()
    print(f"[SUCCESS] Written to {full_table}: {row_count:,} rows")
    return row_count

## 1. Ingest Customers

In [ ]:
customers_schema = StructType([
    StructField("customer_id", StringType(), False),
    StructField("first_name", StringType(), True),
    StructField("last_name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("phone", StringType(), True),
    StructField("date_of_birth", StringType(), True),
    StructField("gender", StringType(), True),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("country", StringType(), True),
    StructField("postal_code", StringType(), True),
    StructField("segment", StringType(), True),
    StructField("registration_date", StringType(), True),
    StructField("is_active", StringType(), True),
])

df_customers_raw = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")  # Use explicit schema in Bronze
    .schema(customers_schema)
    .load(f"{source_folder}customers.csv")
)

df_customers_bronze = add_audit_columns(df_customers_raw, "customers")

# Incremental filter: only new/updated records since watermark
if load_mode == "incremental":
    df_customers_bronze = df_customers_bronze.filter(
        col("registration_date") > watermark_date
    )

write_mode = "append" if load_mode == "incremental" else "overwrite"
write_to_bronze(df_customers_bronze, "customers", write_mode)

df_customers_bronze.printSchema()

## 2. Ingest Products

In [ ]:
products_schema = StructType([
    StructField("product_id", StringType(), False),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("subcategory", StringType(), True),
    StructField("brand", StringType(), True),
    StructField("unit_price", StringType(), True),
    StructField("cost_price", StringType(), True),
    StructField("stock_quantity", StringType(), True),
    StructField("weight_kg", StringType(), True),
    StructField("is_active", StringType(), True),
    StructField("created_date", StringType(), True),
])

df_products_raw = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .schema(products_schema)
    .load(f"{source_folder}products.csv")
)

df_products_bronze = add_audit_columns(df_products_raw, "products")
write_to_bronze(df_products_bronze, "products", "overwrite")

## 3. Ingest Orders

In [ ]:
orders_schema = StructType([
    StructField("order_id", StringType(), False),
    StructField("customer_id", StringType(), True),
    StructField("order_date", StringType(), True),
    StructField("ship_date", StringType(), True),
    StructField("delivery_date", StringType(), True),
    StructField("status", StringType(), True),
    StructField("payment_method", StringType(), True),
    StructField("shipping_method", StringType(), True),
    StructField("shipping_cost", StringType(), True),
    StructField("discount_amount", StringType(), True),
    StructField("subtotal", StringType(), True),
    StructField("tax_amount", StringType(), True),
    StructField("total_amount", StringType(), True),
    StructField("coupon_code", StringType(), True),
    StructField("channel", StringType(), True),
])

df_orders_raw = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .schema(orders_schema)
    .load(f"{source_folder}orders.csv")
)

df_orders_bronze = add_audit_columns(df_orders_raw, "orders")

if load_mode == "incremental":
    df_orders_bronze = df_orders_bronze.filter(
        col("order_date") > watermark_date
    )

write_mode = "append" if load_mode == "incremental" else "overwrite"
write_to_bronze(df_orders_bronze, "orders", write_mode)

## 4. Ingest Order Items

In [ ]:
order_items_schema = StructType([
    StructField("order_item_id", StringType(), False),
    StructField("order_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("quantity", StringType(), True),
    StructField("unit_price", StringType(), True),
    StructField("discount_pct", StringType(), True),
    StructField("line_total", StringType(), True),
])

df_items_raw = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .schema(order_items_schema)
    .load(f"{source_folder}order_items.csv")
)

df_items_bronze = add_audit_columns(df_items_raw, "order_items")

if load_mode == "incremental":
    # Join with orders to get order_date for incremental filtering
    df_orders_ref = spark.table(f"{BRONZE_SCHEMA}.orders").select("order_id", "order_date")
    df_items_bronze = (
        df_items_bronze
        .join(df_orders_ref, "order_id", "inner")
        .filter(col("order_date") > watermark_date)
        .drop("order_date")
    )

write_mode = "append" if load_mode == "incremental" else "overwrite"
write_to_bronze(df_items_bronze, "order_items", write_mode)

## 5. Bronze Layer Validation

In [ ]:
print("=" * 60)
print("BRONZE LAYER INGESTION SUMMARY")
print("=" * 60)

tables = ["customers", "products", "orders", "order_items"]
for table in tables:
    try:
        count = spark.table(f"{BRONZE_SCHEMA}.{table}").count()
        print(f"  {table:20s}: {count:>10,} rows")
    except Exception as e:
        print(f"  {table:20s}: ERROR - {e}")

print("=" * 60)
print(f"Ingestion completed at: {INGESTION_TS}")
print(f"Load mode: {load_mode}")

## 6. Delta Table Optimization

Optimize Delta files and Z-order on key columns to improve query performance for downstream Silver notebook.

In [ ]:
# Optimize Delta tables — compacts small files and applies Z-ordering
# This improves read performance for Silver transformation queries

spark.sql(f"OPTIMIZE {BRONZE_SCHEMA}.orders ZORDER BY (customer_id, order_date)")
spark.sql(f"OPTIMIZE {BRONZE_SCHEMA}.order_items ZORDER BY (order_id, product_id)")
spark.sql(f"OPTIMIZE {BRONZE_SCHEMA}.customers ZORDER BY (customer_id)")
spark.sql(f"OPTIMIZE {BRONZE_SCHEMA}.products ZORDER BY (product_id)")

print("Delta OPTIMIZE completed — small files compacted, Z-ordering applied.")